In [1]:
pip install choix numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import json
import ast 

import numpy as np
import choix


In [3]:
def load_dicts_to_df(
    directory: str | Path,
    pattern: str = "*",
) -> pd.DataFrame:
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    rows: list[dict[str, Any]] = []
    for path in sorted(directory.glob(pattern)):
        if not path.is_file() or path.name.startswith("."):
            continue
        suffix = path.suffix.lower()
        data = None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
        else:
            text = path.read_text(encoding="utf-8")
            data = ast.literal_eval(text)

        if not isinstance(data, dict):
            raise TypeError(f"{path} does not contain a dict (got {type(data).__name__})")
        rows.append(data)

    return pd.json_normalize(rows)



In [4]:
df = load_dicts_to_df("/home/eje/git/coding-agent-leaderboard/results", pattern="*.json")

In [5]:
df.columns

Index(['benchmark.name', 'benchmark.repo', 'benchmark.num_tasks',
       'benchmark.url', 'harness.name', 'harness.skills', 'harness.is_oss',
       'harness.url', 'model.name', 'model.repo', 'model.is_oss',
       'model.num_params', 'model.precision', 'model.url', 'environment.name',
       'environment.config.path', 'environment.config.name',
       'environment.config.version', 'environment.config.ref',
       'environment.config.registry_url', 'environment.config.registry_path',
       'environment.config.overwrite', 'environment.config.download_dir',
       'environment.config.task_names',
       'environment.config.exclude_task_names', 'environment.config.n_tasks',
       'environment.url', 'metrics.n_tasks', 'metrics.n_errors',
       'metrics.score', 'metrics.n_input_tokens', 'metrics.n_cache_tokens',
       'metrics.n_output_tokens', 'metrics.n_total_tokens',
       'metrics.agent_time_seconds', 'metrics.total_time_seconds',
       'metrics.cost_usd', 'metrics.mean_input_toke

In [6]:
df.shape

(22, 45)

In [7]:
df[['harness.name','model.name','metrics.score']].loc[df['metrics.score'] > .5]

,harness.name,model.name,metrics.score
0,Claude Code,Opus 4.6,0.633
1,Claude Code,Sonnet 4.6,0.557
2,Claude Code,Opus 4.8,0.698
3,OpenCode,Opus 4.8,0.781
5,Codex,GPT 5.5 - high,0.604
11,Claude Code,Opus 4.8,0.868
12,OpenCode,Opus 4.8,0.834
13,Claude Code,Sonnet 4.6,0.796
14,Codex,GPT 5.5 - high,0.798
15,Claude Code,Qwen3.6-35B-A3B-NVFP4,0.632


In [8]:
from typing import Any
def prepare_ranking_data(df: pd.DataFrame,
                         catcol: str | list[str],
                         metcol: str,
                         descending: bool = False) -> tuple[list[tuple[int, int]], list[Any]]:
    ndata = df.shape[0]
    if ndata < 2:
        raise ValueError("Not enough data to prepare ranking comparisons")
    catcol = catcol if isinstance(catcol, list) else [catcol]
    ncat = len(catcol)
    tcols = catcol + [metcol]
    t = list(df[tcols].itertuples(index=False, name=None))
    metvals = [x[-1] for x in t]
    if ncat > 1:
        catvals = [x[:-1] for x in t]
    else:
        # single category column: categories are just the values of the column
        catvals = [x[0] for x in t]
    # map item categories to unique integers
    umap = dict([(y,x) for x,y in enumerate(sorted(set(catvals)))])
    compvals = [(umap[c],m) for c,m in zip(catvals, metvals)]
    comps = []
    for i in range(ndata):
        ic, im = compvals[i]
        for j in range(i):
            jc, jm = compvals[j]
            # ignore items with same metric value
            if im == jm: continue
            # pairs are always of form (winner, loser)
            if im > jm:
                comps.append((jc, ic) if descending else (ic, jc))
            else:
                comps.append((ic, jc) if descending else (jc, ic))
    return comps, sorted(umap.keys())


In [9]:
comps, cats = prepare_ranking_data(df, 'harness.name', 'metrics.score')

In [10]:
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
print("Latent strength parameters:")
for name, strength in zip(cats, params):
    print(f"  {name:>11}: {strength:+.3f}")


Latent strength parameters:
  Claude Code: +0.518
        Codex: +1.025
     OpenClaw: -0.709
     OpenCode: -0.113
           Pi: -0.530
    Qwen Code: -0.191


In [11]:
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {cats[idx]}")



Ranking:
  1. Codex
  2. Claude Code
  3. OpenCode
  4. Qwen Code
  5. Pi
  6. OpenClaw
